# Detección del reloj con YOLO + lectura sobre el recorte

In [8]:
import warnings; warnings.filterwarnings("ignore")
import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image, ImageDraw
import timm
from timm.scheduler.cosine_lr import CosineLRScheduler
from ultralytics import YOLO

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"; print(device)

CACHE, RES = "cache", "resultados"
meta = pd.read_csv(f"{CACHE}/meta.csv")
M = {s: meta[meta.subset == s].reset_index(drop=True) for s in ["train", "val", "test"]}
NUM_WORKERS = 6
SEED = 23

def fijar_semilla(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

cuda


In [9]:
def evaluate_clock(yh, ym, ph, pm, tol=2):
    yh, ym, ph, pm = (np.asarray(v).astype(int) for v in (yh, ym, ph, pm))
    valid = ph >= 0
    hour_ok = valid & (ph == yh)
    dm = np.abs(pm - ym) % 60; dm = np.minimum(dm, 60 - dm)
    min_ok = valid & (dm <= tol)
    dt = np.abs(np.where(valid, ph * 60 + pm, 0) - (yh * 60 + ym)) % 720; dt = np.minimum(dt, 720 - dt)
    return {"hour_acc": 100 * hour_ok.mean(), "min_acc": 100 * min_ok.mean(),
            "full_acc": 100 * (hour_ok & min_ok).mean(), "exact_acc": 100 * (hour_ok & (pm == ym)).mean(),
            "mae": dt.mean()}

def guardar_pred(nombre, split, m, ph, pm, logp=None):
    df = pd.DataFrame({"idx": m.idx.values, "hour": m.hour.values, "minute": m.minute.values,
                       "pred_hour": np.asarray(ph), "pred_minute": np.asarray(pm)})
    df.to_csv(f"{RES}/{nombre}_{split}.csv", index=False)
    r = evaluate_clock(df.hour, df.minute, df.pred_hour, df.pred_minute)
    print(f"{nombre:28s} {split:5s} | hora {r['hour_acc']:6.2f} | min±2 {r['min_acc']:6.2f} | "
          f"completa {r['full_acc']:6.2f} | MAE {r['mae']:6.1f}")
    return r

## Detección

In [10]:
# YOLO11x de COCO ya trae la clase 74 "clock"; si no detecta nada se prueba YOLO-World
def best_box(res, W, H, keep=None):
    best, best_s, n = None, -1.0, 0
    for b in res.boxes:
        if keep is not None and int(b.cls) not in keep:
            continue
        n += 1
        x0, y0, x1, y1 = b.xyxy[0].tolist()
        s = float(b.conf) * np.sqrt(max(x1 - x0, 1) * max(y1 - y0, 1) / (W * H))
        if s > best_s:
            best, best_s = (x0, y0, x1, y1), s
    return best, n

def square_crop(img, box, margin=0.12, size=512):
    x0, y0, x1, y1 = box
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    side = max(x1 - x0, y1 - y0) * (1 + 2 * margin)
    return img.crop((int(cx - side / 2), int(cy - side / 2), int(cx + side / 2), int(cy + side / 2))).resize((size, size), Image.Resampling.BICUBIC)

In [11]:
if not os.path.exists(f"{CACHE}/det.csv"):
    coco = YOLO("yolo11x.pt")
    world = YOLO("yolov8x-worldv2.pt")
    world.set_classes(["analog clock", "wristwatch", "clock face", "clock"])
    os.makedirs(f"{CACHE}/crop", exist_ok=True)
    rows, t0 = [], time.time()
    for start in range(0, len(meta), 16):
        chunk = meta.iloc[start:start + 16]
        imgs = [Image.open(f"{CACHE}/orig/{r.split}_{r.idx}.jpg").convert("RGB") for r in chunk.itertuples()]
        results = coco.predict(imgs, conf=0.10, imgsz=1024, verbose=False)
        for img, res, r in zip(imgs, results, chunk.itertuples()):
            W, H = img.size
            box, n = best_box(res, W, H, keep={74}); det = "coco"
            if box is None:
                box, n = best_box(world.predict(img, conf=0.10, imgsz=1024, verbose=False)[0], W, H); det = "world"
            if box is None:
                det = "ninguno"
                crop = Image.open(f"{CACHE}/full/{r.split}_{r.idx}.jpg")          # sin detección: imagen completa
            else:
                crop = square_crop(img, box)
            crop.save(f"{CACHE}/crop/{r.split}_{r.idx}.jpg", quality=95)
            rows.append({"split": r.split, "idx": r.idx, "detector": det, "n_cajas": n,
                         "area": 0.0 if box is None else (box[2] - box[0]) * (box[3] - box[1]) / (W * H),
                         "box": "" if box is None else ",".join(f"{v:.1f}" for v in box)})
        if start % 1600 == 0: print(f"{start}/{len(meta)} ({time.time() - t0:.0f}s)")
    pd.DataFrame(rows).to_csv(f"{CACHE}/det.csv", index=False)
    del coco, world; torch.cuda.empty_cache()

det = pd.read_csv(f"{CACHE}/det.csv").merge(meta, on=["split", "idx"])
print(det.detector.value_counts(normalize=True).round(3))
tab = det.groupby("clock_type").agg(n=("idx", "size"), detectado=("detector", lambda d: 100 * (d != "ninguno").mean()),
                                    varios=("n_cajas", lambda x: 100 * (x > 1).mean()), area_mediana=("area", lambda a: 100 * a.median()))
tab.round(1)

detector
coco       0.948
world      0.036
ninguno    0.016
Name: proportion, dtype: float64


,n,detectado,varios,area_mediana
clock_type,,,,
Alarm/Desk clocks,1501,99.2,57.0,15.7
ETC,1893,95.8,51.8,13.7
Graphic/Illustrated,893,93.5,52.6,19.5
Post clocks,999,100.0,62.2,2.8
Tower clocks,1913,99.8,67.4,0.9
Wall clocks,4046,99.1,40.5,6.6
Wristwatches,1238,98.9,54.8,24.6


In [ ]:
ej = det[det.detector != "ninguno"].sample(6, random_state=1)
fig, axs = plt.subplots(2, 6, figsize=(6.75, 2.6), constrained_layout=True)
for j, r in enumerate(ej.itertuples()):
    img = Image.open(f"{CACHE}/orig/{r.split}_{r.idx}.jpg").convert("RGB")
    ImageDraw.Draw(img).rectangle([float(v) for v in r.box.split(",")], outline=(255, 0, 0), width=max(3, img.width // 150))
    axs[0, j].imshow(img); axs[0, j].set_title(f"{r.hour}:{r.minute:02d} {r.clock_type}", fontsize=6)
    axs[1, j].imshow(Image.open(f"{CACHE}/crop/{r.split}_{r.idx}.jpg"))
    axs[0, j].set_axis_off(); axs[1, j].set_axis_off()
plt.show()

## ConvNext sobre los recortes

In [13]:
def hflip_label(h, m):
    t = (720 - (60 * h + m)) % 720
    return t // 60, t % 60

class ClockDataset(Dataset):
    def __init__(self, meta, folder, transform=None, hflip=0.0):
        self.meta = meta.reset_index(drop=True)
        self.folder = folder
        self.transform = transform
        self.hflip = hflip

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, idx):
        r = self.meta.iloc[idx]
        image = Image.open(f"{CACHE}/{self.folder}/{r.split}_{r.idx}.jpg").convert("RGB")
        h, m = int(r.hour), int(r.minute)
        if self.hflip > 0 and torch.rand(1).item() < self.hflip:
            image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            h, m = hflip_label(h, m)
        if self.transform:
            image = self.transform(image)
        return {"X": image, "y": torch.tensor(60 * h + m)}

def make_transform(res, mean, std, train=False):
    if not train:
        return transforms.Compose([transforms.Resize((res, res), interpolation=transforms.InterpolationMode.BICUBIC),
                                   transforms.ToTensor(), transforms.Normalize(mean, std)])
    return transforms.Compose([
        transforms.RandomResizedCrop(res, scale=(0.65, 1.0), ratio=(0.85, 1.18), interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.RandomApply([transforms.RandomRotation(10)], p=0.5),
        transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.3, 0.05)], p=0.8),
        transforms.RandomGrayscale(0.1),
        transforms.RandomApply([transforms.GaussianBlur(5, (0.1, 1.5))], p=0.2),
        transforms.ToTensor(), transforms.Normalize(mean, std)])

def loss_fn(pred, y, smoothing=0.05):
    pred = pred.float()
    return (F.cross_entropy(pred[:, :12], y // 60, label_smoothing=smoothing)
            + F.cross_entropy(pred[:, 12:72], y % 60, label_smoothing=smoothing))

def joint_logp(pred):
    pred = pred.float()
    return (F.log_softmax(pred[:, :12], 1)[:, :, None] + F.log_softmax(pred[:, 12:72], 1)[:, None, :]).reshape(-1, 720)

FLIP_PERM = torch.tensor([60 * hflip_label(t // 60, t % 60)[0] + hflip_label(t // 60, t % 60)[1] for t in range(720)])

def predict(model, loader, tta=False):
    model.eval()
    LP = []
    with torch.no_grad():
        for batch in loader:
            X = batch["X"].to(device)
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=device == "cuda"):
                lp = joint_logp(model(X))
                if tta:
                    lf = joint_logp(model(torch.flip(X, dims=[3])))
                    back = torch.empty_like(lf); back[:, FLIP_PERM.to(device)] = lf
                    lp = lp + back
            LP.append(lp.float().cpu())
    LP = torch.cat(LP).numpy()
    t = LP.argmax(1)
    return t // 60, t % 60, LP

def exp(model, device, train_loader, val_loader, optimizer, scheduler, epochs=15, ckpt="model.pth"):
    step, best, hist = 0, -1, []
    vm = val_loader.dataset.meta
    for epoch in range(epochs):
        t0 = time.time()
        print(f'Epoch {epoch}: train:', end=' ')
        model.train(); trbatches, trloss = 0, 0
        for batch in train_loader:
            X, y = batch['X'].to(device), batch['y'].to(device); trbatches += 1
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=device == "cuda"):
                pred = model(X)
            loss = loss_fn(pred, y); trloss += loss.item()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); optimizer.zero_grad(); step += 1
            scheduler.step_update(num_updates=step)
        trloss /= trbatches
        ph, pm, _ = predict(model, val_loader)
        r = evaluate_clock(vm.hour, vm.minute, ph, pm)
        print(f"loss {trloss:.4f} val: hora {r['hour_acc']:.2f}% min±2 {r['min_acc']:.2f}% completa {r['full_acc']:.2f}% "
              f"MAE {r['mae']:.1f} ({time.time() - t0:.0f}s)")
        hist.append({"epoch": epoch, "loss": trloss, **r})
        score = r["full_acc"] + r["hour_acc"] / 100
        if score > best:
            best = score
            torch.save(model.state_dict(), ckpt)
    return pd.DataFrame(hist)

def entrenar(model_name, folder, nombre, res=384, epochs=12, lr=1e-4, batch_size=16):
    fijar_semilla()
    model = timm.create_model(model_name, pretrained=True, num_classes=72, drop_path_rate=0.1).to(device)
    cfg = timm.data.resolve_data_config(model.pretrained_cfg)
    kw = dict(num_workers=NUM_WORKERS, pin_memory=True)
    train_loader = DataLoader(ClockDataset(M["train"], folder, make_transform(res, cfg["mean"], cfg["std"], True), 0.5),
                              batch_size=batch_size, shuffle=True, drop_last=True, generator=torch.Generator().manual_seed(SEED), **kw)
    ev = make_transform(res, cfg["mean"], cfg["std"])
    val_loader = DataLoader(ClockDataset(M["val"], folder, ev), batch_size=2 * batch_size, **kw)
    test_loader = DataLoader(ClockDataset(M["test"], folder, ev), batch_size=2 * batch_size, **kw)
    head = {id(p) for p in model.get_classifier().parameters()}
    optimizer = torch.optim.AdamW([{"params": [p for p in model.parameters() if id(p) not in head], "lr": lr},
                                   {"params": [p for p in model.parameters() if id(p) in head], "lr": 10 * lr}], weight_decay=0.05)
    scheduler = CosineLRScheduler(optimizer, t_initial=epochs * len(train_loader), warmup_t=len(train_loader),
                                  warmup_lr_init=1e-6, lr_min=1e-6, t_in_epochs=False)
    hist = exp(model, device, train_loader, val_loader, optimizer, scheduler, epochs, ckpt=f"{nombre}.pth")
    model.load_state_dict(torch.load(f"{nombre}.pth", weights_only=True))
    for s, dl in [("val", val_loader), ("test", test_loader)]:
        guardar_pred(nombre, s, dl.dataset.meta, *predict(model, dl))
        guardar_pred(nombre + "_tta", s, dl.dataset.meta, *predict(model, dl, tta=True))
    del model, optimizer; torch.cuda.empty_cache()
    return hist

In [14]:
# misma configuración que timm_convnext_full del experimento 3: solo cambia la entrada (recorte)
hist_crop = entrenar("convnext_small.fb_in22k_ft_in1k_384", "crop", "timm_convnext_crop")

Epoch 0: train: loss 6.0074 val: hora 24.64% min±2 35.96% completa 8.19% MAE 136.8 (64s)
Epoch 1: train: loss 4.4126 val: hora 59.95% min±2 69.98% completa 45.59% MAE 56.3 (69s)
Epoch 2: train: loss 3.3772 val: hora 72.47% min±2 78.01% completa 61.00% MAE 38.2 (71s)
Epoch 3: train: loss 2.9092 val: hora 77.77% min±2 82.18% completa 69.42% MAE 32.9 (71s)
Epoch 4: train: loss 2.5957 val: hora 79.53% min±2 83.15% completa 71.43% MAE 31.3 (69s)
Epoch 5: train: loss 2.3356 val: hora 80.74% min±2 84.27% completa 73.43% MAE 31.0 (66s)
Epoch 6: train: loss 2.1169 val: hora 82.26% min±2 85.55% completa 75.52% MAE 27.0 (66s)
Epoch 7: train: loss 1.9417 val: hora 82.42% min±2 85.79% completa 76.00% MAE 27.3 (68s)
Epoch 8: train: loss 1.7938 val: hora 82.10% min±2 86.28% completa 75.68% MAE 27.7 (67s)
Epoch 9: train: loss 1.6864 val: hora 82.74% min±2 85.79% completa 75.84% MAE 27.0 (67s)
Epoch 10: train: loss 1.6084 val: hora 81.94% min±2 85.87% completa 75.52% MAE 28.1 (68s)
Epoch 11: train: los

## Resumen imagen vs recorte

In [15]:
filas = []
for n, entrada in [("timm_convnext_full", "completa"), ("timm_convnext_full_tta", "completa"),
                   ("timm_convnext_crop", "recorte"), ("timm_convnext_crop_tta", "recorte")]:
    if os.path.exists(f"{RES}/{n}_test.csv"):
        v, t = pd.read_csv(f"{RES}/{n}_val.csv"), pd.read_csv(f"{RES}/{n}_test.csv")
        filas.append({"modelo": n, "entrada": entrada,
                      "val_full": evaluate_clock(v.hour, v.minute, v.pred_hour, v.pred_minute)["full_acc"],
                      **evaluate_clock(t.hour, t.minute, t.pred_hour, t.pred_minute)})
pd.DataFrame(filas).set_index("modelo").round(2)

,entrada,val_full,hour_acc,min_acc,full_acc,exact_acc,mae
modelo,,,,,,,
timm_convnext_full,completa,33.15,54.93,64.27,44.37,19.00,76.32
timm_convnext_full_tta,completa,36.60,57.73,66.46,47.57,21.82,71.55
timm_convnext_crop,recorte,76.00,78.43,81.67,71.45,36.29,32.44
timm_convnext_crop_tta,recorte,78.09,80.54,82.41,73.93,39.17,29.45
